Checkpoint 2: Data preprocessing + Basic data exploration and summary statistics

Data Preprocessing for the raw data from the Storm Events Database:

In [1]:
import pandas as pd
from datetime import datetime
import string

# Create the original database from the csv of the first year we're using. (1991)
storm_events_df = pd.read_csv('Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d1991_c20250520.csv')

# Go through the folder containing the rest of the CSVs and add their data onto the complete database one by one.
for i in range(1992, 2020):
    year_storms_csv = f"Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d{str(i)}_c20250520.csv"
    year_storms_df = pd.read_csv(year_storms_csv)
    storm_events_df = pd.concat([storm_events_df, year_storms_df], ignore_index=True)

# 2020 is different because the csv file was made at a different time so it has a different naming convention.
final_year = pd.read_csv('Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d2020_c20240620.csv')
storm_events_df = pd.concat([storm_events_df, final_year], ignore_index=True)

# Function uses the BEGIN_YEARMONTH and BEGIN_DAY columns to create a datetime for when the storm took place.
def find_datetime(column1, column2):
    final_column = []
    for i in range(0, len(storm_events_df)):
        year_month = str(storm_events_df.loc[i, column1])
        year = year_month[0:4]
        month = year_month[4:]
        day = str(storm_events_df.loc[i, column2])
        if len(day) == 1:
            day = "0" + day
        new_string = year + "-" + month + "-" + day
        final_datetime = pd.to_datetime(new_string, format = '%Y-%m-%d', errors ='coerce')
        final_column.append(final_datetime)
    return pd.Series(final_column)

# Use the function above to create the EVENT_DATE column, and create columns for the total deaths and total injuries caused.
storm_events_df['EVENT_DATE'] = find_datetime('BEGIN_YEARMONTH','BEGIN_DAY')
storm_events_df['DEATHS_TOTAL'] = storm_events_df['DEATHS_DIRECT'] + storm_events_df['DEATHS_INDIRECT']
storm_events_df['INJURIES_TOTAL'] = storm_events_df['INJURIES_DIRECT'] + storm_events_df['INJURIES_INDIRECT']

# Create the final database by filtering out a lot of the unnecessary (for our purposes) columns.
storm_events_df = storm_events_df[['YEAR', 'EVENT_ID', 'EVENT_TYPE', 'EVENT_DATE', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'INJURIES_TOTAL',
                  'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DEATHS_TOTAL', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS']]

storm_events_df

/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_87493/3051443342.py:9: DtypeWarning: Columns (26,48) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_87493/3051443342.py:9: DtypeWarning: Columns (26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_87493/3051443342.py:9: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_87493/3051443342.py:9: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_87493/30514

,YEAR,EVENT_ID,EVENT_TYPE,EVENT_DATE,INJURIES_DIRECT,INJURIES_INDIRECT,INJURIES_TOTAL,DEATHS_DIRECT,DEATHS_INDIRECT,DEATHS_TOTAL,DAMAGE_PROPERTY,DAMAGE_CROPS
0,1991,9985808,Thunderstorm Wind,1991-06-01,0,0,0,0,0,0,0,0
1,1991,9985809,Thunderstorm Wind,1991-06-01,0,0,0,0,0,0,0,0
2,1991,9985810,Hail,1991-06-01,0,0,0,0,0,0,0,0
3,1991,9985811,Hail,1991-06-01,0,0,0,0,0,0,0,0
4,1991,9985812,Hail,1991-06-02,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1502479,2020,919277,Thunderstorm Wind,2020-08-10,0,0,0,0,0,0,NaN,NaN
1502480,2020,904958,Tornado,2020-06-02,0,0,0,0,0,0,0.00K,0.00K
1502481,2020,904282,Thunderstorm Wind,2020-06-08,0,0,0,0,0,0,NaN,0.00K
1502482,2020,896642,Hail,2020-06-02,0,0,0,0,0,0,NaN,0.00K


In [5]:
print(storm_events_df['YEAR'].count())
print(storm_events_df['EVENT_ID'].count())
print(storm_events_df['EVENT_TYPE'].count())
print(storm_events_df['EVENT_DATE'].count())
print(storm_events_df['INJURIES_DIRECT'].count())
print(storm_events_df['INJURIES_INDIRECT'].count())
print(storm_events_df['INJURIES_TOTAL'].count())
print(storm_events_df['DEATHS_DIRECT'].count())
print(storm_events_df['DEATHS_INDIRECT'].count())
print(storm_events_df['DEATHS_TOTAL'].count())
print(storm_events_df['DAMAGE_PROPERTY'].count())
print(storm_events_df['DAMAGE_CROPS'].count())

1502484
1502484
1502484
1502484
1502484
1502484
1502484
1502484
1502484
1502484
967858
856799


In [33]:
# Function for converting the damage costs into numerical values and removing inconsistencies.
def standardize_costs(value):
    if type(value) is str:
        units = value[len(value) - 1]
        if units == 'h' or units == 'H':
            number = float(value[0:len(value) - 1])
            return (100 * number)
        elif units == 'k' or units == 'K':
            if len(value) == 1:
                number = 0
            else:
                number = float(value[0:len(value) - 1])
            return (1000 * number)
        elif units == 'm' or units == 'M':
            if len(value) == 1:
                number = 0
            else:
                number = float(value[0:len(value) - 1])
            return (1000000 * number)
        elif units == 'b' or units == 'B':
            number = float(value[0:len(value) - 1])
            return (1000000000 * number)
        elif units == 't' or units == 'T':
            number = float(value[0:len(value) - 1])
            return (1000000000000 * number)
        elif units == '?':
            number = float(value[0:len(value) - 1])
            return number
        else:
            number = float(value)
            return number
    else:
        return value

# Replace NaN values with 0 in the columns.
storm_events_df['DAMAGE_PROPERTY'].fillna(0)
storm_events_df['DAMAGE_CROPS'].fillna(0)

storm_events_df['DAMAGE_PROPERTY'] = storm_events_df['DAMAGE_PROPERTY'].apply(standardize_costs)
storm_events_df['DAMAGE_CROPS'] = storm_events_df['DAMAGE_CROPS'].apply(standardize_costs)

In [34]:
storm_events_df['DAMAGE_PROPERTY'].value_counts()

DAMAGE_PROPERTY
0.0           611070
5000.0         49590
10000.0        38318
1000.0         38148
2000.0         29309
               ...  
41000000.0         1
81000000.0         1
8570000.0          1
642000.0           1
266000.0           1
Name: count, Length: 2141, dtype: int64